# 01 · Data preparation

**Purpose.** Download COCO val2017, create the seeded held-out (tuning) / test split, inspect POPE, and pre-compute category co-occurrence statistics for the error analysis.

| | |
|---|---|
| **Proposal objective** | Objective 2 (datasets for evaluation) |
| **Inputs** | internet |
| **Outputs** | `results/splits.json`, `results/data/*.json|csv` |
| **Runtime (T4)** | ~10 min (≈1 GB download, cached on Drive) |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, DRIVE_PROJECT], check=True)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from dvd.config import load_config, save_selected, results_dir
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache
from dvd import paths

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

## 1. COCO val2017
Zips are cached in `data/downloads/` on Drive (persistent); images are extracted to the fast local disk.

In [ ]:
from dvd.data.coco import prepare_coco, CocoData, make_splits
prepare_coco(cfg)
coco = CocoData(cfg)
n_inst = len(coco.coco.getAnnIds(iscrowd=False))
print(f"{len(coco.image_ids)} images, {n_inst} non-crowd instances, {len(coco.cat_ids)} categories")

In [ ]:
cat_counts = pd.Series({coco.cat_id_to_name[c]: len(coco.coco.getAnnIds(catIds=[c], iscrowd=False)) for c in coco.cat_ids})
cat_counts.sort_values(ascending=False).plot.bar(figsize=(16, 4), title="Instances per category (val2017)"); plt.show()
results_dir(cfg, "data")
cat_counts.sort_values(ascending=False).to_csv(results_dir(cfg, "data") / "category_counts.csv", header=["instances"])

## 2. Held-out / test split
The held-out split is used **only** for choosing the crop margin and the decision thresholds. All reported numbers come from the test split. The split is created once and frozen.

In [ ]:
splits = make_splits(cfg, coco.image_ids)
HELDOUT, TEST = splits["heldout"], splits["test"]
assert not set(HELDOUT) & set(TEST)
def split_stats(ids):
    g = coco.gts(ids)
    n = [int((~v["iscrowd"]).sum()) for v in g.values()]
    return {"images": len(ids), "instances": int(np.sum(n)), "inst/img": float(np.mean(n)), "empty images": int(np.sum(np.array(n) == 0))}
pd.DataFrame({"held-out": split_stats(HELDOUT), "test": split_stats(TEST)})

## 3. POPE (verifier evaluated in isolation)
Three negative-sampling settings, 3000 questions each, 50 % yes.

In [ ]:
from dvd.data.pope import iter_pope
rows = []
for s in cfg["pope"]["settings"]:
    items = list(iter_pope(cfg, s, max_n=cfg["pope"]["max_per_setting"]))
    rows.append({"setting": s, "questions": len(items), "yes_fraction": np.mean([i["label"] == "yes" for i in items]),
                 "unique_objects": len({i["object"] for i in items}), "example": items[0]["question"]})
    del items
pope_stats = pd.DataFrame(rows); pope_stats

In [ ]:
pope_stats.to_csv(results_dir(cfg, "data") / "pope_stats.csv", index=False)

## 4. Category co-occurrence (for hallucination-style FP analysis in notebook 10)
Li et al. report that hallucinated objects concentrate on classes that co-occur with objects truly present. We store, for each class, its 10 most frequent co-occurring classes.

In [ ]:
from dvd.analysis import cooccurrence_topk
cooc = cooccurrence_topk(coco, k=10)
save_json({str(k): sorted(v) for k, v in cooc.items()}, results_dir(cfg, "data") / "cooccurrence_top10.json")
for c in [1, 3, 62]:
    print(coco.cat_id_to_name[c], "->", [coco.cat_id_to_name[x] for x in sorted(cooc[c])])

**Next:** `02_detector_agent_baseline`.